# Moduł 15: Duże modele językowe (LLM) i fine-tuning

LLM to **duże modele Transformer** (miliardy parametrów) pretrenowane na ogromnych zbiorach tekstu.

## Spis treści
1. Enkodery vs dekodery — BERT vs GPT
2. Tokenizacja BPE (Byte Pair Encoding)
3. Pretraining: cele uczenia (MLM, NSP, Span Corruption, Denoising)
4. Scaling laws
5. Prompt engineering
6. Fine-tuning i transfer learning
7. RLHF (Reinforcement Learning from Human Feedback)
8. Modele Encoder-Decoder w praktyce (T5, BART, MarianMT)
9. **Zadania**

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Enkodery vs dekodery

### Modele enkoderowe (BERT)
- Widzą **cały kontekst** (lewo i prawo)
- Self-attention **bez maski**
- Cel: **rozumienie** tekstu
- Zastosowania: klasyfikacja, NER, odpowiadanie na pytania
- Pretraining: Masked Language Modeling (MLM) — zgaduj zamaskowane słowa

```
Wejście: [CLS] Ala [MASK] kota [SEP]
BERT: → → → → →
Cel: ↓ ma ↓
```

### Modele dekoderowe (GPT)
- Widzą tylko **lewy kontekst** (causal mask)
- Cel: **generowanie** tekstu
- Pretraining: Next Token Prediction — przewiduj następne słowo
- Zastosowania: generowanie, chatboty, rozumowanie

```
Wejście: Ala ma kota
GPT: → → →
Predykcja: ma kota .
```

### Porównanie

| Cecha | Enkoder (BERT) | Dekoder (GPT) |
|---|---|---|
| Kontekst | Dwukierunkowy | Tylko w lewo |
| Cel pretreningu | MLM (maskowanie) | Next token |
| Generowanie | Słabe | Dobre |
| Rozumienie | Dobre | (przy dużej skali) |
| Przykłady | BERT, RoBERTa | GPT-2/3/4, LLaMA |

## 2. Tokenizacja BPE (Byte Pair Encoding)

LLM nie tokenizują tekstu na słowa ani znaki — używają **subword tokenization** (BPE).

### Algorytm BPE:
1. Zacznij od pojedynczych znaków jako tokeny
2. Znajdź **najczęstszą parę** sąsiednich tokenów
3. Połącz je w nowy token
4. Powtarzaj aż do żądanego rozmiaru słownika

### Przykład:
```
Korpus: "low lower lowest"
Krok 0: l o w _ l o w e r _ l o w e s t
Krok 1: lo w _ lo w e r _ lo w e s t ← (l,o) najczęstsza para
Krok 2: low _ low e r _ low e s t ← (lo,w) najczęstsza
Krok 3: low _ lowe r _ lowe s t ← (low,e) najczęstsza
...
```

### Zalety:
- Rzadkie słowa → rozbijane na znane fragmenty ("unhappiness" → "un" + "happiness")
- Częste słowa → jeden token ("the" = token 1)
- Brak problemu OOV (out-of-vocabulary)

In [ ]:
# Uproszczona implementacja BPE
from collections import Counter

def get_pairs(word_list):
 """Zlicz pary sąsiednich tokenów."""
 pairs = Counter()
 for word, freq in word_list.items():
 symbols = word.split()
 for i in range(len(symbols) - 1):
 pairs[(symbols[i], symbols[i+1])] += freq
 return pairs

def merge_pair(pair, word_list):
 """Połącz najczęstszą parę."""
 new_word_list = {}
 bigram = ' '.join(pair)
 replacement = ''.join(pair)
 for word, freq in word_list.items():
 new_word = word.replace(bigram, replacement)
 new_word_list[new_word] = freq
 return new_word_list

# Korpus z częstościami
corpus = {
 'l o w </w>': 5,
 'l o w e r </w>': 2,
 'l o w e s t </w>': 3,
 'n e w </w>': 4,
 'n e w e r </w>': 1
}

print("=== Symulacja BPE ===")
print(f"\nPoczątkowy słownik: {set(' '.join(corpus.keys()).split())}")

num_merges = 8
for i in range(num_merges):
 pairs = get_pairs(corpus)
 if not pairs:
 break
 best_pair = max(pairs, key=pairs.get)
 corpus = merge_pair(best_pair, corpus)
 print(f"\nMerge {i+1}: {best_pair[0]} + {best_pair[1]} → {''.join(best_pair)} (freq: {pairs[best_pair]})")

print(f"\nKońcowy stan tokenów:")
for word, freq in corpus.items():
 print(f" {word} (×{freq})")

## 3. Pretraining — cele uczenia

### A) Next Token Prediction (GPT)

Model przewiduje następny token na podstawie poprzednich:
$$P(w_t | w_1, w_2, ..., w_{t-1})$$

Loss: **cross-entropy** dla każdego tokenu.

### B) Masked Language Modeling — MLM (BERT)

Losowo **maskuj** ~15% tokenów, model musi je odgadnąć:

Z tych 15%:
- 80% → zamień na `[MASK]`
- 10% → zamień na losowy token
- 10% → zostaw bez zmian

**Dlaczego nie 100% [MASK]?** Bo `[MASK]` nie występuje w fine-tuningu — model musi też radzić sobie z normalnymi tokenami.

**Loss:** CrossEntropy **tylko na zamaskowanych pozycjach**:
$$\mathcal{L}_{\text{MLM}} = -\sum_{i \in \text{masked}} \log P(w_i | \mathbf{w}_{\setminus i})$$

### C) Next Sentence Prediction — NSP (BERT)

Drugi cel pretreningu BERT — rozumienie relacji między zdaniami:

1. Weź parę zdań (A, B)
2. W 50% przypadków B jest **rzeczywistym następnym zdaniem** → etykieta `IsNext`
3. W 50% B to **losowe zdanie** z korpusu → etykieta `NotNext`
4. Model klasyfikuje z wyjścia `[CLS]` tokenu

```
Input: [CLS] Ala ma kota. [SEP] Kot je ryby. [SEP] → IsNext
Input: [CLS] Ala ma kota. [SEP] Pogoda jest ładna. [SEP] → NotNext
```

$$\mathcal{L}_{\text{BERT}} = \mathcal{L}_{\text{MLM}} + \mathcal{L}_{\text{NSP}}$$

**UWAGA:** RoBERTa (2019) wykazała, że **NSP nie pomaga** (a nawet szkodzi). Nowsze modele używają tylko MLM lub wariantów (np. Sentence Order Prediction w ALBERT).

### D) Span Corruption (T5)

T5 maskuje **ciągłe fragmenty** (spans) zamiast pojedynczych tokenów:

```
Input: "The <X> walks in the <Y> and plays <Z>"
Target: "<X> cute dog <Y> park <Z> fetch"
```

- Sentinel tokens `<X>`, `<Y>`, `<Z>` oznaczają zamaskowane fragmenty
- Decoder musi odtworzyć zawartość spanów

### E) Denoising (BART)

BART stosuje różne **szumy** na wejście i rekonstruuje oryginał:

| Szum | Opis |
|------|------|
| Token masking | Jak MLM — zamień tokeny na `[MASK]` |
| Token deletion | Usuń losowe tokeny (model musi zgadnąć pozycje) |
| Sentence permutation | Pomieszaj kolejność zdań |
| Text infilling | Zamień fragmenty na jeden `[MASK]` (różne długości) |

### F) Self-supervised Learning

Wszystkie powyższe podejścia są **self-supervised** — etykiety generowane są z samego tekstu (nie potrzeba ręcznych adnotacji).

Dlaczego to działa? Model musi zrozumieć **gramatykę, semantykę, relacje logiczne** by przewidywać brakujące/następne słowa.

## 4. Scaling Laws

Prawa skalowania (Kaplan et al., 2020) — loss modelu spada **potęgowo** ze wzrostem:

$$L(N) \propto N^{-\alpha_N}$$
$$L(D) \propto D^{-\alpha_D}$$
$$L(C) \propto C^{-\alpha_C}$$

Gdzie:
- $N$ — liczba parametrów modelu
- $D$ — wielkość zbioru danych
- $C$ — budżet obliczeniowy (FLOPs)

### Wnioski:
- Większy model + więcej danych = lepszy wynik (bez optuna, bez inżynierii cech)
- **Chinchilla** (Hoffmann et al., 2022): optymalnie $D \approx 20 \times N$ (20 tokenów na parametr)

In [ ]:
# Wizualizacja scaling laws (symulacja)
params = np.logspace(6, 11, 50) # od 1M do 100B parametrów
loss = 10 * params ** (-0.076) # przybliżony power law

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss vs parametry (log-log)
axes[0].loglog(params, loss)
axes[0].set_xlabel("Liczba parametrów")
axes[0].set_ylabel("Test Loss")
axes[0].set_title("Scaling Law: Loss vs Model Size")
axes[0].grid(True, alpha=0.3)

# Znane modele
models = {
 'GPT-2': 1.5e9,
 'GPT-3': 175e9,
 'LLaMA-7B': 7e9,
 'LLaMA-70B': 70e9
}

sizes = list(models.values())
names = list(models.keys())

axes[1].barh(names, [s/1e9 for s in sizes], color=['#3498db', '#e74c3c', '#2ecc71', '#f39c12'])
axes[1].set_xlabel("Parametry (miliardy)")
axes[1].set_title("Rozmiary znanych LLM")
axes[1].set_xscale('log')

plt.tight_layout()
plt.show()

## 5. Prompt Engineering

Zamiast fine-tunować model, można go sterować przez **prompt** (instrukcję).

### Strategie:

**Zero-shot** — bez przykładów:
```
Klasyfikuj sentyment: "Ten film jest wspaniały!" → pozytywny/negatywny
```

**Few-shot** — z przykładami:
```
Klasyfikuj sentyment:
"Świetna zabawa" → pozytywny
"Strata czasu" → negatywny 
"Film roku!" → pozytywny
"Nudne i długie" → ?
```

**Chain-of-Thought** — rozumowanie krok po kroku:
```
Q: Ala ma 5 jabłek, dała 3 Oli, kupiła 2. Ile ma?
Myślmy krok po kroku:
1. Ala zaczyna z 5 jabłkami
2. Daje 3 → 5-3 = 2
3. Kupuje 2 → 2+2 = 4
Odpowiedź: 4
```

## 6. Fine-tuning i transfer learning

### Podejścia:

**a) Full fine-tuning** — trenuj wszystkie parametry na nowym zadaniu.
- Wymaga dużo GPU
- Ryzyko catastrophic forgetting

**b) Feature extraction** — zamroź bazowy model, trenuj tylko nową głowicę.
- Szybki, mało GPU
- Gorszy wynik

**c) LoRA (Low-Rank Adaptation)** — dodaj małe adaptowalne macierze.
$$W' = W + \Delta W = W + BA$$
Gdzie $B \in \mathbb{R}^{d \times r}$, $A \in \mathbb{R}^{r \times d}$, $r \ll d$

- Trenuje ~0.1% parametrów
- Prawie tak dobry jak full fine-tuning

### Pipeline fine-tuningu:
1. Weź pretrenowany model (np. BERT, LLaMA)
2. Dodaj task-specific head (np. Linear do klasyfikacji)
3. Trenuj na swoim zbiorze danych
4. Ewaluuj na zbiorze testowym

In [ ]:
# Symulacja LoRA — dodawanie low-rank macierzy
import torch
import torch.nn as nn

class LoRALinear(nn.Module):
 """Linear layer z LoRA adaptation."""
 def __init__(self, in_features, out_features, rank=4):
 super().__init__()
 # Oryginalne wagi (zamrożone)
 self.weight = nn.Parameter(torch.randn(out_features, in_features), requires_grad=False)
 self.bias = nn.Parameter(torch.zeros(out_features), requires_grad=False)

 # LoRA: niskoprangowe macierze (trenowalne!)
 self.lora_A = nn.Parameter(torch.randn(rank, in_features) * 0.01)
 self.lora_B = nn.Parameter(torch.zeros(out_features, rank)) # startowo zerowe!

 def forward(self, x):
 # W' = W + B @ A (LoRA)
 return x @ (self.weight + self.lora_B @ self.lora_A).T + self.bias

# Porównanie ilości parametrów
d = 1024
r = 4

full_params = d * d # pełna macierz
lora_params = d * r + r * d # A + B

print(f"Full fine-tuning: {full_params:,} parametrów")
print(f"LoRA (rank={r}): {lora_params:,} parametrów")
print(f"Redukcja: {lora_params/full_params*100:.2f}%")

## 7. RLHF (Reinforcement Learning from Human Feedback)

RLHF to technika, dzięki której ChatGPT jest "pomocny i bezpieczny" (nie tylko mądrze uzupełnia tekst).

### 3 etapy:

**Etap 1: Supervised Fine-Tuning (SFT)**
- Trenuj model na parach (instrukcja → odpowiedź) napisanych przez ludzi

**Etap 2: Reward Model**
- Ludzie **rankują** różne odpowiedzi modelu (A > B > C)
- Trenuj osobny model (Reward Model) przewidujący jakość odpowiedzi

**Etap 3: PPO (Proximal Policy Optimization)**
- Traktuj LLM jako **agenta RL**
- Akcja = wygenerowany token
- Nagroda = score z Reward Modelu
- Optymalizuj politykę (model) by maksymalizować nagrodę

```
Pretrenowany LLM
 ↓ SFT (dane instrukcyjne)
Model SFT
 ↓ ludzie rankują odpowiedzi 
Reward Model (osobna sieć)
 ↓ PPO (RL)
Model RLHF (ChatGPT!)
```

## 8. Modele Encoder-Decoder w praktyce

### 8.1 Przegląd modeli

| Model | Parametry | Pretraining | Mocne strony |
|-------|-----------|------------|-------------|
| **T5** | 60M-11B | Span corruption | Uniwersalny text-to-text |
| **T5v1.1** | j.w. | Bez Dropout, GeGLU | Ulepszona wersja T5 |
| **BART** | 140M-400M | Denoising | Streszczanie, generowanie |
| **mBART** | 610M | Denoising (25 lang) | Wielojęzyczne tłumaczenie |
| **MarianMT** | ~300M | Tłumaczenie (OPUS) | Szybkie, per-para-językowa |
| **mT5** | 300M-13B | Span corruption (101 lang) | Wielojęzyczny T5 |

### 8.2 T5 — Text-to-Text Transfer Transformer

T5 traktuje **każde zadanie NLP** jako text-to-text:

```
# Tłumaczenie:
Input: "translate English to German: That is good."
Target: "Das ist gut."

# Streszczanie:
Input: "summarize: Artykuł o AI..."
Target: "AI zmienia świat..."

# Klasyfikacja:
Input: "sst2 sentence: I love this movie"
Target: "positive"

# Odpowiadanie na pytania:
Input: "question: Who won? context: Poland won the match."
Target: "Poland"
```

### 8.3 BERT jako Encoder (pełny pipeline)

BERT jest najczęściej używany jako **feature extractor / fine-tuned encoder**:

```python
# Typowy pipeline BERT do klasyfikacji:
from transformers import BertTokenizer, BertForSequenceClassification

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

inputs = tokenizer("This movie is great!", return_tensors="pt", 
 padding=True, truncation=True, max_length=128)
outputs = model(**inputs)
# outputs.logits shape: (1, num_labels)
```

**Architektura BERT:**
- 12 warstw Transformer Encoder (BERT-base) / 24 (BERT-large)
- Wejście: `[CLS] tokens [SEP]` (jedna sekwencja) lub `[CLS] A [SEP] B [SEP]` (parowanie)
- `[CLS]` embedding → klasyfikacja
- Token embeddings → NER, token-level tasks

### 8.4 Użycie w HuggingFace

```python
# T5 do streszczania:
from transformers import T5Tokenizer, T5ForConditionalGeneration

tokenizer = T5Tokenizer.from_pretrained("t5-small")
model = T5ForConditionalGeneration.from_pretrained("t5-small")

input_text = "summarize: Machine learning is a field of AI..."
inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)
summary = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(summary[0], skip_special_tokens=True))
```

```python
# MarianMT do tłumaczenia PL→EN:
from transformers import MarianMTModel, MarianTokenizer

tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-pl-en")
model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-pl-en")

texts = ["Ala ma kota.", "Uczenie maszynowe jest fascynujące."]
inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
translated = model.generate(**inputs)
results = tokenizer.batch_decode(translated, skip_special_tokens=True)
```

In [ ]:
# =========================================
# Symulacja MLM (Masked Language Modeling) z NSP
# =========================================
import torch
import torch.nn as nn
import numpy as np

def simulate_mlm_masking(tokens, mask_prob=0.15, vocab_size=1000, mask_token_id=103):
 """Symulacja strategii maskowania BERT (80/10/10)."""
 masked_tokens = tokens.clone()
 labels = torch.full_like(tokens, -100) # -100 = ignoruj w loss

 # Losuj pozycje do zamaskowania (pomijając specjalne tokeny na pozycji 0 i -1)
 num_to_mask = max(1, int(len(tokens) * mask_prob))
 mask_indices = torch.randperm(len(tokens) - 2)[:num_to_mask] + 1

 for idx in mask_indices:
 labels[idx] = tokens[idx] # zapamiętaj oryginalny token
 r = torch.rand(1).item()
 if r < 0.8:
 masked_tokens[idx] = mask_token_id # [MASK]
 elif r < 0.9:
 masked_tokens[idx] = torch.randint(0, vocab_size, (1,)).item() # losowy
 # else: zostaw bez zmian (10%)

 return masked_tokens, labels

# Demo
torch.manual_seed(42)
tokens = torch.tensor([101, 50, 23, 67, 89, 45, 12, 102]) # [CLS] ... [SEP]
print("Oryginalne tokeny:", tokens.tolist())

masked, labels = simulate_mlm_masking(tokens, mask_prob=0.3)
print("Po maskowaniu: ", masked.tolist(), " (103 = [MASK])")
print("Labels (target): ", labels.tolist(), " (-100 = ignoruj)")

# =========================================
# NSP (Next Sentence Prediction) — demo
# =========================================
def create_nsp_example(sentence_a, sentence_b, is_next):
 """Tworzy przykład NSP."""
 # [CLS] A [SEP] B [SEP]
 input_ids = [101] + sentence_a + [102] + sentence_b + [102]
 token_type_ids = [0] * (len(sentence_a) + 2) + [1] * (len(sentence_b) + 1)
 label = 1 if is_next else 0
 return {
 "input_ids": input_ids,
 "token_type_ids": token_type_ids,
 "nsp_label": label
 }

# Pozytywny przykład (rzeczywista następna sekwencja)
pos = create_nsp_example([50, 23, 67], [89, 45], is_next=True)
# Negatywny przykład (losowa sekwencja)
neg = create_nsp_example([50, 23, 67], [12, 34], is_next=False)

print(f"\nNSP positive: {pos}")
print(f"NSP negative: {neg}")

# =========================================
# Porównanie rozmiarów modeli
# =========================================
models_info = {
 "BERT-base": {"params": 110e6, "layers": 12, "hidden": 768, "heads": 12},
 "BERT-large": {"params": 340e6, "layers": 24, "hidden": 1024, "heads": 16},
 "RoBERTa-base": {"params": 125e6, "layers": 12, "hidden": 768, "heads": 12},
 "T5-small": {"params": 60e6, "layers": 6, "hidden": 512, "heads": 8},
 "T5-base": {"params": 220e6, "layers": 12, "hidden": 768, "heads": 12},
 "BART-base": {"params": 140e6, "layers": 6, "hidden": 768, "heads": 12},
 "MarianMT": {"params": 300e6, "layers": 6, "hidden": 512, "heads": 8},
}

print("\n--- Porównanie modeli encoder / encoder-decoder ---")
print(f"{'Model':<15} {'Params':>10} {'Layers':>7} {'Hidden':>7} {'Heads':>6}")
print("-" * 50)
for name, info in models_info.items():
 print(f"{name:<15} {info['params']/1e6:>8.0f}M {info['layers']:>7} {info['hidden']:>7} {info['heads']:>6}")

---
---
## Zadania do samodzielnego rozwiązania

---

### Zadanie 1: BPE od zera (średnie)

Zaimplementuj pełny algorytm BPE:
1. Wejście: tekst (np. 10 zdań po angielsku)
2. Tokenizuj na znaki
3. Iteracyjnie scalaj najczęstsze pary (num_merges=20)
4. Wypisz końcowy słownik
5. Napisz funkcję `encode(text)` → lista tokenów
6. Napisz funkcję `decode(tokens)` → tekst

In [ ]:
# Zadanie 1: Pełna implementacja BPE
# TWÓJ KOD TUTAJ

### Zadanie 2: MLM (Masked Language Model) — prosty BERT (trudne)

Zaimplementuj uproszczoną wersję BERT MLM:
1. Weź mały korpus (10+ zdań)
2. Tokenizuj, zbuduj słownik
3. Losowo maskuj 15% tokenów
4. Model: Embedding → TransformerEncoder → Linear(d_model, vocab_size)
5. Loss: CrossEntropy **tylko na zamaskowanych pozycjach**
6. Po treningu: podaj zdanie z [MASK], sprawdź predykcje

In [ ]:
# Zadanie 2: Mini BERT MLM
# TWÓJ KOD TUTAJ

### Zadanie 3: Prompt engineering — analiza (łatwe)

Tym zadaniem jest **analiza teoretyczna** (bez kodu):

1. Napisz **3 przykłady** few-shot promptów do klasyfikacji sentymentu (pozytywny/negatywny)
2. Napisz **2 przykłady** chain-of-thought promptów do prostych zadań matematycznych
3. Dla każdego prompt'a wyjaśnij: dlaczego ta strategia jest lepsza niż zero-shot?

**Twoja odpowiedź do Zadania 3:**

(wpisz tutaj)

### Zadanie 4: LoRA — eksperyment (średnie)

1. Stwórz prosty model liniowy (d=256)
2. Zaimplementuj LoRA adapter z różnymi rangami: r=1, 4, 16, 64
3. Dla każdego rangu policz:
 - Liczbę parametrów do trenowania
 - % redukcji vs full fine-tuning
4. Narysuj wykres: ranga vs % parametrów
5. Jak ranga wpływa na ekspresywność? Kiedy r jest za małe?

In [ ]:
# Zadanie 4: LoRA eksperyment
# TWÓJ KOD TUTAJ

### Zadanie 5: Perplexity — miara jakosci modelu jezykowego (srednie)

$$\text{PPL} = \exp\left(-\frac{1}{N}\sum_{i=1}^{N} \log p(x_i | x_{<i})\right)$$

1. Zaimplementuj funkcje `compute_perplexity(logits, targets)` 
2. Wygeneruj sztuczne logity (model losowy vs "dobry" model)
3. Pokaz, ze lepsze predykcje = nizsza perplexity
4. Co oznacza PPL = 1? Co oznacza PPL = vocab_size?

In [ ]:
# Zadanie 5: Perplexity
import torch
import torch.nn.functional as F
import numpy as np

def compute_perplexity(logits, targets):
    # TWOJ KOD TUTAJ
    pass

# Test z losowymi danymi
# vocab_size = 1000
# seq_len = 50
# logits = torch.randn(seq_len, vocab_size)
# targets = torch.randint(0, vocab_size, (seq_len,))
# print(f"Random model PPL: {compute_perplexity(logits, targets):.2f}")

### Zadanie 6: Tokenizer porownanie (srednie)

Porownaj 3 tokenizery HuggingFace na tym samym tekscie polskim:
1. `bert-base-multilingual-cased` (WordPiece)
2. `allegro/herbert-base-cased` (BPE)
3. `speakleash/Bielik-11B-v2.3-Instruct` (SentencePiece)

Dla kazdego:
- Ile tokenow generuje?
- Jakie subwordy tworzy?
- Ktory lepiej radzi sobie z polskimi slowami?

Tekst testowy: "Przetwarzanie jezyka naturalnego w jezyku polskim jest wymagajace ze wzgledu na fleksje."

In [ ]:
# Zadanie 6: Porownanie tokenizerow
# Uwaga: wymaga zainstalowanych modeli HuggingFace
# from transformers import AutoTokenizer

tekst = "Przetwarzanie jezyka naturalnego w jezyku polskim jest wymagajace ze wzgledu na fleksje."
# TWOJ KOD TUTAJ

---

## Dodatek OAI: Ćwiczenia w stylu olimpiadowym

Na 2. etapie III OAI będziesz mógł użyć **Bielik-11B-v3.0-Instruct** (lokalnie, bez internetu).

### Powiązane zadania OAI:
- **I OAI**: Dependency parsing z HerBERT (pretrained model)
- **II OAI**: Halucynacje (wykrywanie błędów LLM)
- **II OAI finał**: Stylizacja tłumaczenia (LLM jako narzędzie)
- **II OAI finał**: Machine Unlearning (modyfikacja wyuczonego modelu)

### Kluczowe dla olimpiady:
- Bielik używa formatu **ChatML** (`<|im_start|>`, `<|im_end|>`)
- Ładowanie: `AutoModelForCausalLM.from_pretrained(..., torch_dtype=torch.bfloat16)`
- Po użyciu **zwolnij GPU** (`del model; gc.collect(); torch.cuda.empty_cache()`)
- Szczegóły → **Moduł 16: Bielik praktyka**

### Ćwiczenia:
- **OAI-13A**: Wykrywanie halucynacji — porównaj odpowiedź LLM z faktem
- **OAI-13B**: Prompt engineering — few-shot classification po polsku
- **OAI-13C**: Machine Unlearning — selektywne "zapominanie" klasy

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# === OAI-13A: Wykrywanie halucynacji LLM ===
print("=== Wykrywanie halucynacji (wzorzec: II OAI 'Halucynuj') ===")

# Symulacja: mamy fakt i odpowiedź modelu
facts = [
 "Stolica Polski to Warszawa.",
 "Python został stworzony przez Guido van Rossuma.",
 "Mount Everest ma 8849 metrów wysokości.",
 "Woda wrze w 100°C przy normalnym ciśnieniu.",
]

model_answers = [
 "Stolica Polski to Warszawa.", # OK
 "Python został stworzony przez Elona Muska.", # HALUCYNACJA
 "Mount Everest ma 8849 metrów wysokości.", # OK
 "Woda wrze w 50°C przy normalnym ciśnieniu.", # HALUCYNACJA
]

def detect_hallucination_simple(fact, answer, threshold=0.8):
 """
 Prosta detekcja halucynacji na podstawie podobieństwa tokenów.
 Na olimpiadzie użyjesz Bielika lub embeddingów!
 """
 fact_tokens = set(fact.lower().split())
 answer_tokens = set(answer.lower().split())

 if len(fact_tokens) == 0:
 return True

 overlap = len(fact_tokens & answer_tokens) / len(fact_tokens)
 return overlap < threshold

print("Detekcja halucynacji (prosta metoda tokenowa):")
for fact, answer in zip(facts, model_answers):
 is_hallucination = detect_hallucination_simple(fact, answer)
 status = " HALUCYNACJA" if is_hallucination else " OK"
 print(f" {status}: '{answer[:50]}...'")

print("\n Na olimpiadzie: użyj embeddingów (cosine similarity) lub Bielika do porównania")

# === OAI-13B: Few-shot classification po polsku ===
print("\n=== Few-shot Classification (wzorzec: II OAI, III OAI) ===")

def prepare_few_shot_dataset(examples_per_class=3):
 """
 Przygotuj dane do few-shot classification.
 Na olimpiadzie dostaniesz mało danych treningowych → few-shot jest kluczowy!
 """
 dataset = {
 "sport": [
 "Legia wygrała mecz 2:0",
 "Iga Świątek zwyciężyła w turnieju",
 "Polska drużyna olimpijska zdobyła złoto",
 "Lewandowski strzelił hat-tricka",
 "Maraton w Warszawie zgromadził 10 tysięcy biegaczy",
 ],
 "technologia": [
 "Nowy iPhone ma lepszy procesor",
 "Sztuczna inteligencja rewolucjonizuje medycynę",
 "Tesla wypuściła nowy model samochodu elektrycznego",
 "ChatGPT osiągnął 100 milionów użytkowników",
 "Kwantowe komputery rozwiązują coraz trudniejsze problemy",
 ],
 "nauka": [
 "Naukowcy odkryli nową cząstkę elementarną",
 "Teleskop Jamesa Webba sfotografował odległą galaktykę",
 "Badania nad CRISPR przynoszą przełomowe wyniki",
 "Opublikowano wyniki eksperymentu z fuzją jądrową",
 "Polscy matematycy udowodnili ważne twierdzenie",
 ],
 }

 train_data, test_data = [], []
 for label, texts in dataset.items():
 for text in texts[:examples_per_class]:
 train_data.append((text, label))
 for text in texts[examples_per_class:]:
 test_data.append((text, label))

 return train_data, test_data

train, test = prepare_few_shot_dataset(examples_per_class=3)
print(f" Few-shot: {len(train)} przykładów treningowych, {len(test)} testowych")
print(f" Klasy: {set(label for _, label in train)}")

# Generuj prompt few-shot
prompt_examples = "\n".join([f"Tekst: '{t}' → Kategoria: {l}" for t, l in train])
print(f"\n Prompt few-shot (fragment):")
print(f" {prompt_examples[:200]}...")

print("\n Bielik świetnie radzi sobie z few-shot classification po polsku!")

# === OAI-13C: Machine Unlearning (selektywne zapominanie) ===
print("\n=== Machine Unlearning (wzorzec: II OAI finał) ===")

# Koncepcja: wytrenowany model musi "zapomnieć" jedną klasę
# Metoda: fine-tuning z odwróconymi gradientami dla wybranej klasy

class SimpleClassifier(nn.Module):
 def __init__(self, input_dim=20, num_classes=5):
 super().__init__()
 self.net = nn.Sequential(
 nn.Linear(input_dim, 64),
 nn.ReLU(),
 nn.Linear(64, num_classes)
 )

 def forward(self, x):
 return self.net(x)

# 1. Wytrenuj model na wszystkich klasach
np.random.seed(42)
torch.manual_seed(42)

n_per_class = 50
X_all, y_all = [], []
for c in range(5):
 X_c = np.random.randn(n_per_class, 20) + c * 0.5
 X_all.append(X_c)
 y_all.extend([c] * n_per_class)

X_all = torch.FloatTensor(np.vstack(X_all))
y_all = torch.LongTensor(y_all)

model_full = SimpleClassifier()
optimizer = torch.optim.Adam(model_full.parameters(), lr=1e-3)

for epoch in range(50):
 optimizer.zero_grad()
 loss = nn.CrossEntropyLoss()(model_full(X_all), y_all)
 loss.backward()
 optimizer.step()

# Accuracy przed unlearning
with torch.no_grad():
 preds = model_full(X_all).argmax(dim=1)
 acc_before = (preds == y_all).float().mean()

 # Accuracy na klasie do zapomnienia (klasa 2)
 mask_forget = (y_all == 2)
 acc_forget_before = (preds[mask_forget] == y_all[mask_forget]).float().mean()

 # Accuracy na pozostałych klasach
 mask_keep = (y_all != 2)
 acc_keep_before = (preds[mask_keep] == y_all[mask_keep]).float().mean()

print(f" PRZED unlearning:")
print(f" Accuracy ogólna: {acc_before:.4f}")
print(f" Acc klasa 2 (forget): {acc_forget_before:.4f}")
print(f" Acc reszta (keep): {acc_keep_before:.4f}")

# 2. Machine Unlearning — fine-tuning z losowymi labelami dla klasy 2
model_unlearn = SimpleClassifier()
model_unlearn.load_state_dict(model_full.state_dict())
optimizer_ul = torch.optim.Adam(model_unlearn.parameters(), lr=5e-4)

for epoch in range(30):
 optimizer_ul.zero_grad()

 # Dla klasy do zapomnienia: losowe labele (zaburzenie)
 y_modified = y_all.clone()
 y_modified[mask_forget] = torch.randint(0, 5, (mask_forget.sum(),))

 # Ważone: normalny loss na keep + zaburzony loss na forget
 loss_keep = nn.CrossEntropyLoss()(model_unlearn(X_all[mask_keep]), y_modified[mask_keep])
 loss_forget = nn.CrossEntropyLoss()(model_unlearn(X_all[mask_forget]), y_modified[mask_forget])
 loss = loss_keep + 0.5 * loss_forget

 loss.backward()
 optimizer_ul.step()

# Accuracy po unlearning
with torch.no_grad():
 preds_ul = model_unlearn(X_all).argmax(dim=1)
 acc_after = (preds_ul == y_all).float().mean()
 acc_forget_after = (preds_ul[mask_forget] == y_all[mask_forget]).float().mean()
 acc_keep_after = (preds_ul[mask_keep] == y_all[mask_keep]).float().mean()

print(f"\n PO unlearning (klasa 2):")
print(f" Accuracy ogólna: {acc_after:.4f}")
print(f" Acc klasa 2 (forget): {acc_forget_after:.4f} (↓ cel: niska)")
print(f" Acc reszta (keep): {acc_keep_after:.4f} (↗ cel: wysoka)")
print(f"\n Machine Unlearning: model 'zapomina' klasę 2, zachowując wiedzę o reszcie")